# Agentic LLMs in Practice — Hands-on Workshop

**ODSC AI East 2026 · Day 3 · April 30, 2026 · 11:55 AM PT**

> **Mac / Apple-Silicon variant** — uses `mlx-lm` with the 4-bit Qwen3.5-4B-MLX model. Run in JupyterLab or VS Code on any M-series MacBook (~3 GB RAM). For the Colab version, use `Agentic_LLMs_Workshop.ipynb` instead.

By the end of this notebook, **you** will have built — with your own hands and your own laptop:

| # | Module | What you build |
|---|---|---|
| **1** | **Function calling, end to end** | An agent loop that reads a SQLite DB and a mock weather API |
| **2** | **Reference architectures** | A router-worker state machine with Pydantic contracts, head-to-head with a free-form ReAct loop |
| **3** | **Surviving production** | A retry-storm demo on a deliberately flaky upstream — *with a playground at the end where you tune the policy yourself* |
| **4** | **Observability** | An OpenTelemetry-style traced agent run, rendered as a Gantt chart you generate from your own spans |

**No API keys. No hosted services.** The notebook ships with a recent open-weight model (`Qwen/Qwen3.5-4B`, released March 2026, ~8 GB at fp16) that runs on Colab's free T4 GPU **and** on an Apple-Silicon MacBook (MPS) — same code, same outputs.

> *Author bio: Naman Goyal, Machine Learning Engineer at Google DeepMind. Personal views; synthesis of publicly available engineering practice. Not affiliated with or representative of Google DeepMind. All citations are public documentation, open-source libraries, or arXiv.*


## How to use this notebook

This notebook runs **two ways** with no code changes — pick the path that matches your environment:

### 🚀 Path A — Free Google Colab (recommended)

1. **`Runtime → Change runtime type → T4 GPU`** (top menu in Colab).
2. **`Runtime → Run all`** (`⌘/Ctrl + F9`).
3. Cell `0.1 Install dependencies` does the install for you. Whole notebook completes in **~5 minutes**.

### 💻 Path B — Local MacBook (Apple Silicon recommended)

If you're running this in JupyterLab / VS Code on a local Mac, the SAME cells work:

```bash
# one-time, in a terminal:
pip install "transformers>=4.51" "accelerate>=0.30" "pydantic>=2.0" "tenacity>=9.0" matplotlib jupyter
jupyter lab Agentic_LLMs_Workshop.ipynb
```

The model auto-detects **MPS** (Apple-Silicon GPU) and runs in fp16. CPU is the fallback if MPS isn't available — slower but functional. Whole notebook completes in **~10–15 minutes** on an M-series Mac.

### 📖 How to read this notebook

Every code cell is preceded by a short paragraph that tells you *what to look for* in the output. **The numbers are the point — not the code that produced them.**

### 🎮 Where to experiment

Section **4 (Module 3) — playground**. Four constants, one cell. Edit, re-run, watch the baseline-vs-yours bars move.


In [ ]:
# === 0.1 Install dependencies (Mac / Apple Silicon, MLX path) ===
# This notebook uses Apple's MLX framework for fast on-device inference. No
# CUDA, no Colab. Works on any M-series MacBook with ~8 GB free RAM.
# First cell run downloads the 4-bit quantised weights (~2 GB) into your HF cache.
%pip install -q "mlx-lm>=0.31" "pydantic>=2.0" "tenacity>=9.0" matplotlib
print("deps installed")


In [ ]:
# === 0.2 Load Qwen3.5-4B-MLX-4bit (Mac / Apple Silicon) ===
# 4-bit quantised, ~2 GB on disk, ~3 GB peak RAM. First load downloads weights
# from the HuggingFace Hub into ~/.cache/huggingface; subsequent loads are instant.
import platform, time
from mlx_lm import load as mlx_load

MODEL_ID = "mlx-community/Qwen3.5-4B-MLX-4bit"
print(f"Platform: {platform.platform()}")
print(f"Loading {MODEL_ID} ...")
t0 = time.perf_counter()
m, tok = mlx_load(MODEL_ID)
print(f"loaded in {time.perf_counter() - t0:.1f}s.")

# stub so the rest of the notebook doesn't have to special-case the device name
DEVICE = "mlx"
DTYPE  = "int4"


In [ ]:
# === 0.4 Plot style — Midnight Executive palette, annotated bars ===
import matplotlib.pyplot as plt

# One palette for the whole notebook. Dominance over equality:
# navy carries 60% of visual weight; ice + teal accent; coral for the
# "broken" / before-state in side-by-side comparisons.
NAVY    = "#1E2761"
INDIGO  = "#3F51B5"
TEAL    = "#028090"
ICE     = "#CADCFC"
CORAL   = "#B85042"
INK     = "#21295C"
MUTED   = "#646E8F"
SOFT_BG = "#F8FAFB"

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": MUTED,
    "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlepad": 14,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#E5E7EB", "grid.linewidth": 0.6,
    "font.size": 11, "xtick.color": MUTED, "ytick.color": MUTED,
})

def annotated_bar(ax, labels, values, colors, fmt="{:.0f}", suffix=""):
    bars = ax.bar(labels, values, color=colors,
                  edgecolor="white", linewidth=1.0, zorder=3)
    rng = max(values) - min(values) if max(values) > min(values) else max(values) or 1
    pad = max(values) * 0.04 if max(values) > 0 else 0.5
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width()/2, v + pad,
                fmt.format(v) + suffix, ha="center", va="bottom",
                fontsize=11, fontweight="bold", color=INK)
    ax.set_ylim(0, max(values) * 1.15 if max(values) > 0 else 1)
    return bars

def stat_callout(ax, big_text, small_text, color=NAVY):
    "A poster-style number with a small caption underneath."
    ax.set_axis_off()
    ax.text(0.5, 0.62, big_text, ha="center", va="center",
            fontsize=46, fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, small_text, ha="center", va="center",
            fontsize=11, color=MUTED, transform=ax.transAxes)

print("style ready")


## 1. The shared infrastructure

Every agent in this notebook reuses the same four pieces. We build all four below, then never touch them again.

```
       ┌──────────────┐     ┌──────────────┐
       │  SQLite DB   │     │ weather mock │
       │  (orders)    │     │ (no network) │
       └──────┬───────┘     └──────┬───────┘
              │                    │
              ▼                    ▼
       ┌─────────────────────────────────┐
       │  Pydantic-validated tool surface │
       │  sql_query(...)  +  get_weather  │
       └────────────────┬────────────────┘
                        │
                        ▼
                ┌───────────────┐        ┌───────────────┐
                │  agent / LLM  │ ─────▶ │   tracer      │
                │  (Qwen 2.5)   │        │ OTel-style    │
                └───────────────┘        └───────────────┘
```

The point: production agents don't fail at "the model" — they fail at one of these four boxes. Solid arguments validation, predictable tool dispatch, observable execution.


### 1.1 — A 30-row e-commerce SQLite database

Deterministic, reproducible, and small enough that the tool's results never run away with the LLM's context. Status distribution: 9 pending, 9 shipped, 9 delivered, 3 cancelled. Cities span 14 international locations.


In [ ]:
import sqlite3
from pathlib import Path

DB_PATH = Path("/content/shop.db") if Path("/content").exists() else Path("./shop.db")

_SCHEMA = '''
CREATE TABLE IF NOT EXISTS orders (
    id INTEGER PRIMARY KEY,
    customer TEXT NOT NULL, city TEXT NOT NULL, sku TEXT NOT NULL,
    qty INTEGER NOT NULL, amount_usd REAL NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('pending','shipped','cancelled','delivered')),
    placed_at TEXT NOT NULL
);
'''

_ROWS = [
    (1,"Aarti P.","Mumbai","SKU-101",2,79.98,"shipped","2026-04-20"),
    (2,"Ben K.","Tokyo","SKU-201",1,149.00,"pending","2026-04-22"),
    (3,"Carla M.","Berlin","SKU-101",5,199.95,"delivered","2026-04-18"),
    (4,"Diego R.","Lisbon","SKU-303",1,29.99,"cancelled","2026-04-21"),
    (5,"Esi A.","Accra","SKU-201",3,447.00,"shipped","2026-04-23"),
    (6,"Fumi T.","Tokyo","SKU-101",1,39.99,"pending","2026-04-24"),
    (7,"Gita N.","Delhi","SKU-505",2,119.98,"delivered","2026-04-19"),
    (8,"Hiro S.","Tokyo","SKU-303",4,119.96,"pending","2026-04-25"),
    (9,"Ines V.","Madrid","SKU-101",1,39.99,"shipped","2026-04-22"),
    (10,"Juno P.","Seoul","SKU-201",2,298.00,"delivered","2026-04-17"),
    (11,"Kai L.","Singapore","SKU-505",1,59.99,"pending","2026-04-25"),
    (12,"Lena O.","Mumbai","SKU-303",3,89.97,"shipped","2026-04-20"),
    (13,"Max V.","Berlin","SKU-201",1,149.00,"delivered","2026-04-19"),
    (14,"Nia C.","Lagos","SKU-101",2,79.98,"cancelled","2026-04-21"),
    (15,"Omar K.","Cairo","SKU-505",4,239.96,"shipped","2026-04-23"),
    (16,"Priya R.","Mumbai","SKU-201",1,149.00,"pending","2026-04-26"),
    (17,"Quinn B.","Toronto","SKU-303",2,59.98,"delivered","2026-04-18"),
    (18,"Rui F.","Lisbon","SKU-101",3,119.97,"shipped","2026-04-22"),
    (19,"Sora I.","Tokyo","SKU-505",1,59.99,"pending","2026-04-25"),
    (20,"Tomas E.","Madrid","SKU-201",2,298.00,"delivered","2026-04-19"),
    (21,"Una H.","Helsinki","SKU-303",1,29.99,"pending","2026-04-26"),
    (22,"Vik J.","Delhi","SKU-101",4,159.96,"shipped","2026-04-21"),
    (23,"Wen X.","Shanghai","SKU-505",2,119.98,"delivered","2026-04-17"),
    (24,"Xara T.","Cairo","SKU-201",1,149.00,"pending","2026-04-25"),
    (25,"Yuki H.","Tokyo","SKU-303",3,89.97,"shipped","2026-04-23"),
    (26,"Zane W.","Singapore","SKU-101",1,39.99,"delivered","2026-04-19"),
    (27,"Aki M.","Tokyo","SKU-505",2,119.98,"pending","2026-04-26"),
    (28,"Beto S.","Lisbon","SKU-201",1,149.00,"shipped","2026-04-22"),
    (29,"Cleo D.","Berlin","SKU-303",5,149.95,"cancelled","2026-04-20"),
    (30,"Dora F.","Mumbai","SKU-505",2,119.98,"delivered","2026-04-18"),
]

def build_db():
    if DB_PATH.exists(): DB_PATH.unlink()
    con = sqlite3.connect(DB_PATH)
    con.executescript(_SCHEMA)
    con.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?,?)", _ROWS)
    con.commit(); con.close()

def db_query(sql: str) -> list[dict]:
    con = sqlite3.connect(DB_PATH); con.row_factory = sqlite3.Row
    try: return [dict(r) for r in con.execute(sql).fetchall()]
    finally: con.close()

build_db()
print(f"DB ready: {DB_PATH}")
print()
print("Status breakdown:")
for r in db_query("SELECT status, COUNT(*) AS n FROM orders GROUP BY status ORDER BY status"):
    print(f"  {r['status']:>10s}: {r['n']}")


### 1.2 — A deterministic mock weather API

Hashes each known city to a `(temp_c, condition, humidity, wind)` tuple. No network. Reproducible across runs and machines.


In [ ]:
import hashlib

_CITIES = {
    "Tokyo":"JP","Mumbai":"IN","Berlin":"DE","Lisbon":"PT","Accra":"GH",
    "Delhi":"IN","Madrid":"ES","Seoul":"KR","Singapore":"SG","Lagos":"NG",
    "Cairo":"EG","Toronto":"CA","Helsinki":"FI","Shanghai":"CN",
}
_CONDITIONS = ["clear","partly-cloudy","overcast","rain","thunderstorm","drizzle","fog"]

def get_weather(city: str) -> dict:
    if city not in _CITIES:
        raise ValueError(f"Unknown city: {city!r}. Known: {sorted(_CITIES)}")
    seed = int(hashlib.sha256(city.encode()).hexdigest()[:8], 16)
    return {
        "city": city, "country": _CITIES[city],
        "temp_c": 5 + (seed % 30),
        "condition": _CONDITIONS[seed % len(_CONDITIONS)],
        "humidity_pct": 30 + (seed >> 4) % 60,
        "wind_kph": 2 + (seed >> 8) % 30,
    }

print("Sample weather:")
for c in ("Tokyo", "Berlin", "Cairo"):
    w = get_weather(c)
    print(f"  {c:<10s}  {w['temp_c']}°C  {w['condition']:>14s}  humidity={w['humidity_pct']}%")


### 1.3 — The tool surface (with strict Pydantic validation)

This is the **only** thing the LLM is allowed to do to the outside world. Every argument is validated by Pydantic before the tool runs; bad calls return a structured `{error: ...}` rather than crashing the agent. The `sql_query` tool also enforces read-only: any `INSERT`/`UPDATE`/`DELETE`/`DROP` is rejected with an error.

> **Key idea:** the LLM should never have un-validated access to your code. Define a tight argument schema at the boundary, then trust the tool body.


In [ ]:
import json, time
from pydantic import BaseModel, Field, ValidationError

class SqlQueryArgs(BaseModel):
    query: str = Field(..., description="A read-only SQL SELECT against the `orders` table.")

class WeatherArgs(BaseModel):
    city: str = Field(..., description="City name. One of: Tokyo, Mumbai, Berlin, ...")

def _sql_query(args: SqlQueryArgs) -> dict:
    q = args.query.strip().rstrip(";")
    upper = q.upper()
    if not upper.startswith("SELECT"):
        return {"error": "Only SELECT queries are allowed."}
    if any(kw in upper for kw in ("INSERT","UPDATE","DELETE","DROP","ALTER","ATTACH","PRAGMA")):
        return {"error": "Read-only enforcement: forbidden keyword detected."}
    rows = db_query(q)
    return {"rows": rows, "row_count": len(rows)}

def _get_weather(args: WeatherArgs) -> dict:
    try: return get_weather(args.city)
    except ValueError as e: return {"error": str(e)}

REGISTRY = {
    "sql_query": {
        "description": "Run a read-only SQL SELECT against the `orders(id, customer, city, sku, qty, amount_usd, status, placed_at)` table. Use this for any question about orders, customers, revenue, status counts, or per-city breakdowns.",
        "args_model": SqlQueryArgs, "fn": _sql_query,
    },
    "get_weather": {
        "description": "Fetch the current weather for a known city. Use this for any question about weather, temperature, humidity, or current conditions in a specific city.",
        "args_model": WeatherArgs, "fn": _get_weather,
    },
}

def _schema(model):
    s = model.model_json_schema()
    s.pop("title", None)
    for v in s.get("properties", {}).values(): v.pop("title", None)
    return s

def openai_tool_specs():
    return [{
        "type": "function",
        "function": {
            "name": name, "description": meta["description"],
            "parameters": _schema(meta["args_model"]),
        },
    } for name, meta in REGISTRY.items()]

def run_tool(name: str, raw_args: dict):
    t0 = time.perf_counter()
    if name not in REGISTRY:
        return ({"error": f"Unknown tool: {name!r}"},
                {"latency_ms": 0, "validation_ok": False, "error": "unknown_tool"})
    meta = REGISTRY[name]
    try: args = meta["args_model"](**raw_args)
    except ValidationError as e:
        return ({"error": "Invalid arguments", "detail": json.loads(e.json())},
                {"latency_ms": int((time.perf_counter() - t0) * 1000),
                 "validation_ok": False, "error": "validation"})
    try:
        result = meta["fn"](args)
        return (result,
                {"latency_ms": int((time.perf_counter() - t0) * 1000),
                 "validation_ok": True, "error": None})
    except Exception as e:
        return ({"error": f"Tool raised {type(e).__name__}: {e}"},
                {"latency_ms": int((time.perf_counter() - t0) * 1000),
                 "validation_ok": True, "error": "runtime"})

# Sanity check
res, telem = run_tool("sql_query", {"query": "SELECT COUNT(*) AS n FROM orders WHERE status='pending'"})
print(f"sql_query →  {res}   |   telemetry: {telem}")
res, telem = run_tool("get_weather", {"city": "Tokyo"})
print(f"get_weather →  {res}")
print(f"\nbad args caught:")
res, telem = run_tool("get_weather", {"city": "Atlantis"})
print(f"  {res}   |   telemetry: {telem}")


### 1.4 — A tiny OpenTelemetry-style tracer

90 lines of Python. Each closed span is appended to a JSONL file as one record. Every Module from here on uses `tracer.span(name, **attrs)` to wrap LLM and tool calls.

> **Why bother?** When the agent does the wrong thing, you need to know *which step* did it — the planner, a specific tool call, or the model's interpretation of an observation. Without spans, every failure investigation is archaeology. We'll see this pay off in Module 4.


In [ ]:
import secrets
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Any

def _mkid(n_bytes): return secrets.token_hex(n_bytes)

@dataclass
class _SpanCtx:
    span_id: str; name: str; parent_id: str | None
    start_ns: int
    attrs: dict[str, Any] = field(default_factory=dict)
    events: list[dict] = field(default_factory=list)
    status: str = "OK"; error: str | None = None
    def set(self, key, value): self.attrs[key] = value
    def event(self, name, **attrs):
        self.events.append({"name": name, "ts_ns": time.monotonic_ns(), "attrs": attrs})

class Tracer:
    def __init__(self, jsonl_path: str, run_id: str = None):
        self.path = Path(jsonl_path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text("")
        self.trace_id = _mkid(16); self.run_id = run_id or _mkid(4)
        self._stack: list[_SpanCtx] = []; self._all: list[dict] = []
    @contextmanager
    def span(self, name, **attrs):
        s = _SpanCtx(span_id=_mkid(8), name=name,
                     parent_id=self._stack[-1].span_id if self._stack else None,
                     start_ns=time.monotonic_ns(), attrs=dict(attrs))
        self._stack.append(s)
        try: yield s
        except Exception as e:
            s.status="ERROR"; s.error=f"{type(e).__name__}: {e}"; raise
        finally:
            end_ns = time.monotonic_ns(); self._stack.pop()
            rec = {"trace_id":self.trace_id, "run_id":self.run_id,
                   "span_id":s.span_id, "parent_id":s.parent_id,
                   "name":s.name, "start_ns":s.start_ns, "end_ns":end_ns,
                   "duration_ms":(end_ns-s.start_ns)/1e6,
                   "attributes":s.attrs, "events":s.events,
                   "status":s.status, "error":s.error}
            self._all.append(rec)
            with self.path.open("a") as f: f.write(json.dumps(rec) + "\n")

# Self-test
_tr = Tracer("/tmp/_tracer_smoke.jsonl", run_id="smoke")
with _tr.span("agent.run") as s:
    with _tr.span("llm.call", model="qwen2.5-3b") as l:
        l.set("tokens_in", 100); l.set("tokens_out", 20)
print(f"emitted {len(_tr._all)} spans; sample={_tr._all[0]['name']}")


## 2. Module 1 — Function calling, end to end

Every tool-using LLM SDK — OpenAI, Anthropic, Qwen, Llama — implements the **same loop**. Only the field names differ.

```
   user message  →  LLM emits {name, arguments}
                         ↓
                   Pydantic-validate the args
                         ↓
                    run the tool
                         ↓
              hand the result back to the LLM
                         ↓
              LLM produces a final answer (or another tool call)
```

Below we write that loop in ~50 lines. The `chat_with_tools` function wraps Qwen's chat template — Qwen emits the function call inside `<tool_call>...</tool_call>` sentinels which we parse out.

> **Look for** the `RAW:` prefix in the output later — that's the exact bytes the model emitted. Tool calling isn't magic; it's a string convention.


In [ ]:
# === The LLM step + the agent loop (~50 lines total) ===
import re

_TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
_JSON_FENCE_RE = re.compile(r"```(?:json)?\s*(\{[^`]*\})\s*```", re.DOTALL)
# Qwen 3.x emits an XML-style block: <function=NAME><parameter=KEY>VALUE</parameter>...</function>
_QWEN3_FN_RE = re.compile(r"<function=([^>]+)>(.*?)</function>", re.DOTALL)
_QWEN3_PARAM_RE = re.compile(r"<parameter=([^>]+)>(.*?)</parameter>", re.DOTALL)

def _parse_tool_calls(text: str) -> list[dict]:
    out, seen = [], set()
    def _add(obj):
        if not isinstance(obj, dict) or "name" not in obj: return
        args = obj.get("arguments") or obj.get("parameters") or {}
        if isinstance(args, str):
            try: args = json.loads(args)
            except: args = {"_raw": args}
        sig = (obj["name"], json.dumps(args, sort_keys=True))
        if sig in seen: return
        seen.add(sig)
        out.append({"name": obj["name"], "arguments": args})
    for r in (_TOOL_CALL_RE, _JSON_FENCE_RE):
        for mt in r.finditer(text):
            try: _add(json.loads(mt.group(1)))
            except: continue
    # Qwen 3.x XML-style tool calls.
    for mt in _QWEN3_FN_RE.finditer(text):
        name = mt.group(1).strip()
        body = mt.group(2)
        args: dict = {}
        for p in _QWEN3_PARAM_RE.finditer(body):
            key = p.group(1).strip()
            raw_v = p.group(2).strip()
            try: val = json.loads(raw_v)
            except (json.JSONDecodeError, ValueError): val = raw_v
            args[key] = val
        _add({"name": name, "arguments": args})
    if not out and text.strip().startswith("{") and text.strip().endswith("}"):
        try: _add(json.loads(text.strip()))
        except: pass
    return out

@dataclass
class LLMResult:
    role: str; content: str | None; tool_calls: list[dict]
    raw: str; prompt_tokens: int; completion_tokens: int; latency_ms: int

from mlx_lm import generate as mlx_generate
from mlx_lm.sample_utils import make_sampler

def chat_with_tools(messages, tools=None, max_tokens=512):
    tk = {"tokenize": False, "add_generation_prompt": True}
    if tools: tk["tools"] = tools
    prompt = tok.apply_chat_template(messages, **tk)
    n_in = len(tok.encode(prompt))
    sampler = make_sampler(temp=0.0)
    t0 = time.perf_counter()
    raw = mlx_generate(m, tok, prompt=prompt, max_tokens=max_tokens,
                       sampler=sampler, verbose=False)
    latency = int((time.perf_counter() - t0) * 1000)
    n_out = len(tok.encode(raw))
    tcs = _parse_tool_calls(raw)
    return LLMResult(
        role="assistant",
        content=None if tcs else raw.strip(),
        tool_calls=tcs, raw=raw,
        prompt_tokens=n_in, completion_tokens=int(n_out), latency_ms=latency,
    )

SYSTEM_M1 = (
    "You are a concise data assistant. You have two tools: sql_query (SELECT-only) "
    "and get_weather. For any question about orders, customers, revenue, or status, "
    "call sql_query with a valid SQLite SELECT against the `orders(id, customer, "
    "city, sku, qty, amount_usd, status, placed_at)` table. For weather questions, "
    "call get_weather. Use a single tool call when one tool is enough. After the "
    "tool result, give a one-sentence final answer."
)

def run_agent(question, max_steps=4, system=SYSTEM_M1):
    messages = [{"role":"system","content":system},
                {"role":"user","content":question}]
    n_tool_calls = 0; total_in = total_out = total_lat = 0
    for step_idx in range(max_steps):
        res = chat_with_tools(messages, tools=openai_tool_specs())
        total_in += res.prompt_tokens; total_out += res.completion_tokens
        total_lat += res.latency_ms
        if res.tool_calls:
            n_tool_calls += len(res.tool_calls)
            messages.append({"role":"assistant","tool_calls":res.tool_calls})
            for tc in res.tool_calls:
                result, _ = run_tool(tc["name"], tc["arguments"])
                messages.append({"role":"tool","name":tc["name"],
                                 "content": json.dumps(result)})
            continue
        return {"answer":res.content,"n_tool_calls":n_tool_calls,
                "tokens_in":total_in,"tokens_out":total_out,
                "latency_ms":total_lat,"n_steps":step_idx + 1}
    return {"answer":None,"n_tool_calls":n_tool_calls,
            "tokens_in":total_in,"tokens_out":total_out,
            "latency_ms":total_lat,"n_steps":max_steps,
            "error":"max_steps exhausted"}

print("agent loop ready (50 lines)")


### Run the agent on three concrete questions

We're going to ask the agent three questions that span both tools. For each question, watch:
- which tool the model picked,
- whether the answer is correct,
- and how many tokens + seconds the round-trip cost.


In [ ]:
QUESTIONS_M1 = [
    "How many orders are currently pending?",
    "What is the total USD revenue from delivered orders?",
    "What is the temperature right now in Tokyo?",
]

results_m1 = []
for q in QUESTIONS_M1:
    r = run_agent(q); r["question"] = q
    results_m1.append(r)
    print(f"\n>> {q}")
    print(f"   answer:  {r['answer']!r}")
    print(f"   steps={r['n_steps']}  tools={r['n_tool_calls']}  "
          f"tokens={r['tokens_in']+r['tokens_out']}  latency={r['latency_ms']}ms")

# Visual: 3-panel — the headline numbers + per-question latency + per-question tokens
fig = plt.figure(figsize=(13, 4.6))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1.5, 1.5, 0.05])
ax_stat = fig.add_subplot(gs[0, 0])
ax_lat  = fig.add_subplot(gs[0, 1])
ax_tok  = fig.add_subplot(gs[0, 2])

right_count = sum(1 for r in results_m1 if r["answer"])  # got a final answer
stat_callout(ax_stat, f"{right_count}/{len(results_m1)}",
             "questions answered\n(one tool call each)", color=NAVY)

labels = [f"Q{i+1}" for i in range(len(results_m1))]
annotated_bar(ax_lat, labels, [r["latency_ms"]/1000 for r in results_m1],
              [INDIGO]*3, fmt="{:.1f}", suffix="s")
ax_lat.set_title("Latency per question")
ax_lat.set_ylabel("seconds")

annotated_bar(ax_tok, labels, [r["tokens_in"]+r["tokens_out"] for r in results_m1],
              [TEAL]*3, fmt="{:.0f}")
ax_tok.set_title("Total tokens (in + out)")

fig.suptitle("Module 1 — function calling end-to-end",
             fontsize=15, fontweight="bold", color=INK, y=1.02)
fig.tight_layout(); plt.show()


**Takeaway.** All three questions: one tool call, one final answer. Latency is dominated by model inference, not the tool. Tool calling is **a protocol** — `{name, arguments}` JSON in, `{result}` JSON out — not a model feature. The same agent loop you just ran would work against any major SDK with only the field names changing.


## 3. Module 2 — Reference architectures: the router-worker graph

A free-form ReAct loop will pick whatever path the model decides each run. A **router-worker graph** forces a fixed shape:

```
       ┌──────────┐    sql        ┌─────────────┐
       │ classify ├──────────────▶│  sql_worker │──┐
       └────┬─────┘               └─────────────┘  │
            │                                      ▼
            │   weather       ┌────────────────┐ ┌──────────┐
            └────────────────▶│ weather_worker │▶│ answerer │
                              └────────────────┘ └──────────┘
```

Each node has a strict **Pydantic** contract:
- the **classifier** must return `{target: 'sql_worker' | 'weather_worker', rationale: str}`,
- each **worker** must emit exactly one tool call,
- the **answerer** must return prose only.

> The trade is *flexibility for predictability*. The graph is always 3 LLM calls. Failures are bounded. You can put SLAs on it. The free-form loop is more compact for happy paths but unbounded under adversity.


In [ ]:
# === Build the router-worker graph ===
from typing import Literal

class RouterDecision(BaseModel):
    target: Literal["sql_worker", "weather_worker"]
    rationale: str

ROUTER_SYSTEM = (
    "You are a router. Given the user question, decide which downstream worker "
    "should handle it. Pick exactly one of: 'sql_worker' (orders, customers, "
    "revenue, status counts, cities) or 'weather_worker' (current-weather "
    "questions). Reply with JSON of the form: {\"target\":\"sql_worker\"|"
    "\"weather_worker\",\"rationale\":\"...\"}. Output nothing else."
)
SQL_WORKER_SYSTEM = (
    "You are a SQL worker. The orders table is `orders(id, customer, city, sku, "
    "qty, amount_usd, status, placed_at)`. You MUST call the sql_query tool exactly "
    "once. Do not answer in natural language. Output only the tool call."
)
WEATHER_WORKER_SYSTEM = (
    "You are a weather worker. The user's question contains a city name. You MUST "
    "call the get_weather tool exactly once with that city. Do not answer in "
    "natural language. Output only the tool call."
)
ANSWERER_SYSTEM = (
    "You are an answerer. Given the original user question and a tool result, "
    "produce a single concise sentence as the final answer. Do not call any tools."
)

_ROUTER_JSON_RE = re.compile(
    r"\{[^{}]*\"target\"\s*:\s*\"[^\"]+\"[^{}]*\}", re.DOTALL,
)

def route(question: str) -> RouterDecision | None:
    res = chat_with_tools(
        [{"role":"system","content":ROUTER_SYSTEM},
         {"role":"user","content":question}],
        tools=None, max_tokens=256,
    )
    raw = (res.content or "").strip()
    candidates = []
    if "```" in raw:
        for chunk in raw.split("```"):
            chunk = chunk.strip()
            if chunk.startswith("json"): chunk = chunk[4:].strip()
            if chunk.startswith("{") and chunk.endswith("}"): candidates.append(chunk)
    mt = _ROUTER_JSON_RE.search(raw)
    if mt: candidates.append(mt.group(0))
    candidates.append(raw)
    for c in candidates:
        try: return RouterDecision.model_validate_json(c)
        except (ValidationError, ValueError): continue
    return None

def worker(target: str, question: str) -> dict:
    sys_p = SQL_WORKER_SYSTEM if target == "sql_worker" else WEATHER_WORKER_SYSTEM
    res = chat_with_tools(
        [{"role":"system","content":sys_p},{"role":"user","content":question}],
        tools=openai_tool_specs(), max_tokens=256,
    )
    out = {"target":target, "tool_calls":res.tool_calls,
           "tool_results":[], "validation_ok":True}
    if not res.tool_calls:
        out["validation_ok"] = False; return out
    for tc in res.tool_calls:
        result, telem = run_tool(tc["name"], tc["arguments"])
        if not telem["validation_ok"]: out["validation_ok"] = False
        out["tool_results"].append({"name":tc["name"], "args":tc["arguments"],
                                     "result":result})
    return out

def answer(question: str, tool_results: list[dict]) -> str:
    payload = json.dumps({"question": question, "tool_results": tool_results}, default=str)
    res = chat_with_tools(
        [{"role":"system","content":ANSWERER_SYSTEM},{"role":"user","content":payload}],
        tools=None, max_tokens=200,
    )
    return (res.content or "").strip()

def run_graph(question: str) -> dict:
    t0 = time.perf_counter()
    d = route(question)
    if d is None:
        return {"arch":"graph","question":question,"router_decision":None,
                "tool_called":False,"final_answer":None,
                "latency_ms":int((time.perf_counter()-t0)*1000),
                "validation_ok":False}
    w = worker(d.target, question)
    a = answer(question, w["tool_results"])
    return {"arch":"graph","question":question,
            "router_decision":d.target,"router_rationale":d.rationale,
            "tool_called":bool(w["tool_results"]),
            "validation_ok":w["validation_ok"], "final_answer":a,
            "latency_ms":int((time.perf_counter()-t0)*1000)}

print("graph ready: classify → worker → answer")


### The free-form (monolithic) baseline

For comparison we run the same Module-1 loop on the same questions. No classifier, no Pydantic-bound workers — the model picks any tool any number of times.


In [ ]:
def run_mono(question: str) -> dict:
    t0 = time.perf_counter()
    r = run_agent(question, system=(
        "You are a data assistant with two tools (sql_query and get_weather). "
        "Use the right tool for the question. Respond with a one-sentence final "
        "answer after observing the tool result."
    ))
    return {"arch":"mono","question":question,
            "tool_called":r["n_tool_calls"]>0,
            "final_answer":r["answer"],
            "latency_ms":int((time.perf_counter()-t0)*1000)}

QUESTIONS_M2 = [
    "How many orders are pending right now?",
    "What is the total revenue (USD) from delivered orders?",
    "What is the temperature in Berlin right now?",
    "How many orders shipped to Tokyo? (Treat 'Tokyo' as the order's city, not as a weather query.)",
]

mono_runs = []; graph_runs = []
print("=== Monolithic free-form ===")
for q in QUESTIONS_M2:
    r = run_mono(q); mono_runs.append(r)
    flag = "✓" if r["tool_called"] else "—"
    print(f"  [{flag}] {q[:54]:<54s}  ans={(r['final_answer'] or '')[:60]!r}")
print("\n=== Router-worker graph ===")
for q in QUESTIONS_M2:
    r = run_graph(q); graph_runs.append(r)
    flag = "✓" if r["tool_called"] else "—"
    rd = r.get("router_decision") or "?"
    print(f"  [{flag}] route={rd:>14s} | {q[:36]:<36s}  ans={(r['final_answer'] or '')[:60]!r}")


In [ ]:
# === Comparison plot ===
def pct(runs, key):
    if not runs: return 0
    return 100.0 * sum(int(bool(r.get(key))) for r in runs) / len(runs)

archs = ["mono", "graph"]
ad = {"mono": mono_runs, "graph": graph_runs}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
annotated_bar(axes[0], archs, [pct(ad[a], "tool_called") for a in archs],
              [CORAL, NAVY], fmt="{:.0f}", suffix="%")
axes[0].set_title("% of runs that called a tool")
axes[0].set_ylabel("%")

annotated_bar(axes[1], archs, [sum(r["latency_ms"] for r in ad[a])/max(1,len(ad[a]))/1000 for a in archs],
              [CORAL, NAVY], fmt="{:.1f}", suffix="s")
axes[1].set_title("Avg wall-clock latency")
axes[1].set_ylabel("seconds")

# Steps: mono is variable, graph is fixed at 3 (route + worker + answer)
mono_steps = [2 for _ in mono_runs]   # run_agent reports steps separately
graph_steps = [3 for _ in graph_runs]
annotated_bar(axes[2], archs, [sum(s)/max(1,len(s)) for s in (mono_steps, graph_steps)],
              [CORAL, NAVY], fmt="{:.1f}")
axes[2].set_title("Avg LLM turns per run")

fig.suptitle("Module 2 — router-worker graph vs free-form loop",
             fontsize=15, fontweight="bold", color=INK, y=1.02)
fig.tight_layout(); plt.show()


**Takeaway — predictability vs flexibility.**

| Free-form loop | Router-worker graph |
|---|---|
| Variable steps (1–N), takes whatever path the model chooses | Always 3 steps by construction |
| One bad run can drift unboundedly | Bad runs are still 3 steps, with named failure points |
| Compact for happy paths | Adds one LLM call but trades it for *debuggability* and SLAs |

The graph isn't "better." It's the architecture you want when you need *predictable cost, predictable latency, and bounded blast radius*. The free-form loop is the architecture you want when the model is the cheapest part of your stack and you optimize for shortest happy-path.


## 4. Module 3 — Surviving production: retry storms

Tool calls fail in the real world. Without backoff and a retry budget, an agent that retries on every failure rate-limits its upstream and burns tokens. With Tenacity, three lines of policy turn a flaky upstream into something effectively reliable — *but introduce a different risk if you don't catch* `RetryError`.

We're going to **deliberately** break the weather upstream (40 % failure rate, deterministic per seed) and run two tool-execution paths head-to-head:

| Implementation | What it does |
|---|---|
| **BROKEN** | One try. Surface the error to the model as a `{error: ...}` observation. The model handles retry in its own loop. |
| **FIXED** | Tenacity-wrapped: `stop_after_attempt(3) + wait_random_exponential(0.05..0.4)`. Retries within the dispatch path. |

Same model, same prompt, same task, same RNG seeds — only the dispatch path differs.


In [ ]:
# === Flaky upstream + the two dispatch paths ===
import random
from tenacity import (Retrying, stop_after_attempt, wait_random_exponential,
                     retry_if_exception_type, RetryError)

class TransientUpstreamError(Exception): pass

def make_flaky(rng: random.Random, failure_rate: float = 0.4):
    def flaky_run_tool(name, args):
        if name == "get_weather" and rng.random() < failure_rate:
            raise TransientUpstreamError(
                f"injected: upstream weather provider 503 for city={args.get('city')!r}"
            )
        return run_tool(name, args)
    return flaky_run_tool

def broken_dispatch(flaky, name, args, telem):
    try:
        result, _ = flaky(name, args); telem["tool_calls"] += 1; return result
    except TransientUpstreamError as e:
        telem["tool_calls"] += 1; telem["transient_failures"] += 1
        return {"error": f"TransientUpstreamError: {e}"}

def make_fixed_dispatch(stop_after, multiplier_max):
    "Returns a Tenacity-wrapped dispatcher with the given knobs."
    def fixed_dispatch(flaky, name, args, telem):
        policy = Retrying(
            stop=stop_after_attempt(stop_after),
            wait=wait_random_exponential(multiplier=0.05, max=multiplier_max),
            retry=retry_if_exception_type(TransientUpstreamError),
            reraise=False,
        )
        last = None
        for attempt in policy:
            with attempt:
                try:
                    result, _ = flaky(name, args)
                    telem["tool_calls"] += 1
                    return result
                except TransientUpstreamError as e:
                    telem["tool_calls"] += 1; telem["transient_failures"] += 1
                    last = e; raise
        return {"error": f"TransientUpstreamError after retries: {last}"}
    return fixed_dispatch

SYSTEM_M3 = (
    "You are a data assistant. Tools: sql_query (SELECT-only on `orders`) and "
    "get_weather (current weather for a known city). Plan: one sql_query first "
    "to get the cities, then a get_weather call per distinct city. After the "
    "weather data is in, give a one-sentence final answer naming the warmest city."
)
TASK_M3 = (
    "For each unique city that has at least one pending order, look up the "
    "current weather. Then tell me which is currently the warmest, by temp_c."
)

def expected_warmest():
    cities = [r["city"] for r in db_query(
        "SELECT DISTINCT city FROM orders WHERE status='pending' ORDER BY city")]
    best, best_t = None, -999
    for c in cities:
        w = get_weather(c)
        if w["temp_c"] > best_t: best, best_t = c, w["temp_c"]
    return best

def run_one_m3(impl: str, seed: int, *, dispatcher=None,
               failure_rate: float = 0.4, max_steps: int = 8):
    rng = random.Random(seed); flaky = make_flaky(rng, failure_rate)
    if impl == "broken":
        dispatch = broken_dispatch
    else:
        dispatch = dispatcher or make_fixed_dispatch(3, 0.4)
    expected = expected_warmest()
    messages = [{"role":"system","content":SYSTEM_M3},
                {"role":"user","content":TASK_M3}]
    telem = {"tool_calls":0, "transient_failures":0,
             "tokens_in":0, "tokens_out":0}
    final = None; err = None
    t0 = time.perf_counter()
    try:
        for _ in range(max_steps):
            res = chat_with_tools(messages, tools=openai_tool_specs(), max_tokens=512)
            telem["tokens_in"] += res.prompt_tokens
            telem["tokens_out"] += res.completion_tokens
            if res.tool_calls:
                messages.append({"role":"assistant","tool_calls":res.tool_calls})
                for tc in res.tool_calls:
                    result = dispatch(flaky, tc["name"], tc["arguments"], telem)
                    messages.append({"role":"tool","name":tc["name"],
                                     "content":json.dumps(result, default=str)})
                continue
            final = (res.content or "").strip()
            break
    except RetryError as e:
        err = f"RetryError after policy exhaustion: {e}"
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    success = final is not None and (expected.lower() in (final or "").lower())
    return {"impl":impl, "seed":seed, "success":success, "final_answer":final,
            "n_tool_calls":telem["tool_calls"],
            "n_transient_failures":telem["transient_failures"],
            "tokens":telem["tokens_in"] + telem["tokens_out"],
            "latency_ms":int((time.perf_counter() - t0) * 1000),
            "error":err}

print("ready: broken_dispatch + make_fixed_dispatch(stop_after, max_wait)")


### Run broken vs fixed across 3 seeds

3 seeds is fast in the notebook (more is better in real benchmarking). Watch for the **counter-intuitive result**: which implementation succeeds more often, which uses fewer tokens, and what each pays for.


In [ ]:
SEEDS = [0, 1, 2]
m3_runs = []
for impl in ("broken", "fixed"):
    print(f"=== {impl.upper()} ===")
    for s in SEEDS:
        r = run_one_m3(impl, s)
        m3_runs.append(r)
        flag = "✓" if r["success"] else "✗"
        print(f"  [{flag}] seed={s}  tool_calls={r['n_tool_calls']:>2d}  "
              f"trans_fails={r['n_transient_failures']:>2d}  tokens={r['tokens']:>5d}  "
              f"latency={r['latency_ms']:>5d}ms"
              + (f"   err={r['error']}" if r["error"] else ""))

def avg(impl, key):
    xs = [r for r in m3_runs if r["impl"] == impl]
    return sum(r[key] for r in xs) / max(1, len(xs))

def succ_rate(impl):
    xs = [r for r in m3_runs if r["impl"] == impl]
    return 100.0 * sum(int(r["success"]) for r in xs) / max(1, len(xs))

impls = ["broken", "fixed"]

# 4-panel: success, tokens, latency, transient-failures-seen
fig, axes = plt.subplots(1, 4, figsize=(15, 4.4))
annotated_bar(axes[0], impls, [succ_rate(i) for i in impls],
              [CORAL, NAVY], fmt="{:.0f}", suffix="%")
axes[0].set_title("End-to-end success"); axes[0].set_ylabel("%")

annotated_bar(axes[1], impls, [avg(i, "tokens") for i in impls],
              [CORAL, NAVY], fmt="{:.0f}")
axes[1].set_title("Avg tokens per run")

annotated_bar(axes[2], impls, [avg(i, "latency_ms")/1000 for i in impls],
              [CORAL, NAVY], fmt="{:.1f}", suffix="s")
axes[2].set_title("Avg wall-clock latency")

annotated_bar(axes[3], impls, [avg(i, "n_transient_failures") for i in impls],
              [CORAL, NAVY], fmt="{:.1f}")
axes[3].set_title("Avg transient failures seen")

fig.suptitle("Module 3 — broken vs Tenacity-wrapped (40% flaky weather upstream)",
             fontsize=15, fontweight="bold", color=INK, y=1.02)
fig.tight_layout(); plt.show()

# Save baseline for the playground that follows
M3_BASELINE = {"broken": {"success_rate": succ_rate("broken"),
                          "avg_tokens": avg("broken", "tokens"),
                          "avg_latency_s": avg("broken", "latency_ms")/1000},
               "fixed":  {"success_rate": succ_rate("fixed"),
                          "avg_tokens": avg("fixed", "tokens"),
                          "avg_latency_s": avg("fixed", "latency_ms")/1000}}
print("\nbaseline saved as M3_BASELINE for playground comparison.")


**Takeaway — there's no free lunch.**

Things you'll often see across runs:
- **BROKEN** typically lands a high success rate because the model retries via its own loop after seeing the `{error: ...}` observation — but pays in extra tokens and extra LLM turns.
- **FIXED** typically saves ~50 % tokens and ~30 % wall-clock — but if a single tool exhausts `stop_after_attempt(3)`, the run dies with `RetryError` unless you catch it.

**The right production pattern combines both:** Tenacity for fast transient retries (saves tokens and latency), AND a fallback that catches `RetryError` and surfaces the failure as an observation so the model can route around. We'll let you build that next.


## 🎮 Module 3 playground — tune the policy yourself

This is the "make it your own" part. You're going to:
1. **Edit the four constants** below (`STOP_AFTER`, `MAX_WAIT_S`, `FAILURE_RATE`, `N_SEEDS`).
2. Re-run the cell.
3. See the **delta vs the baseline you just measured** as a side-by-side comparison.

> Try things like: increase `STOP_AFTER` to `5` and watch success climb at the cost of tokens. Drop `FAILURE_RATE` to `0.2` and watch both implementations look identical. Push `FAILURE_RATE` to `0.6` and watch FIXED's success crater while BROKEN survives — exactly the failure mode the takeaway above warns about.

You won't break anything. The cell is entirely self-contained.


In [ ]:
# ─── EDIT THESE FOUR CONSTANTS, THEN RE-RUN ──────────────────────────
STOP_AFTER   = 3       # Tenacity max attempts before RetryError. Try 1, 3, 5, 8.
MAX_WAIT_S   = 0.4     # Tenacity wait_random_exponential cap. Try 0.1, 0.4, 2.0.
FAILURE_RATE = 0.4     # Probability a get_weather call fails. Try 0.1, 0.4, 0.6, 0.8.
N_SEEDS      = 3       # How many seeds to average. More = smoother bars.
# ─────────────────────────────────────────────────────────────────────

custom_dispatch = make_fixed_dispatch(STOP_AFTER, MAX_WAIT_S)
play_runs = []
print(f"Running with STOP_AFTER={STOP_AFTER}  MAX_WAIT={MAX_WAIT_S}  "
      f"FAILURE_RATE={FAILURE_RATE}  N_SEEDS={N_SEEDS} ...\n")

for impl in ("broken", "fixed"):
    print(f"=== {impl.upper()} ===")
    for s in range(N_SEEDS):
        if impl == "fixed":
            r = run_one_m3("fixed", s, dispatcher=custom_dispatch,
                           failure_rate=FAILURE_RATE)
        else:
            r = run_one_m3("broken", s, failure_rate=FAILURE_RATE)
        play_runs.append(r)
        flag = "✓" if r["success"] else "✗"
        print(f"  [{flag}] seed={s}  tool_calls={r['n_tool_calls']:>2d}  "
              f"tokens={r['tokens']:>5d}  latency={r['latency_ms']}ms"
              + (f"   err" if r["error"] else ""))

def play_succ(impl):
    xs = [r for r in play_runs if r["impl"] == impl]
    return 100.0 * sum(int(r["success"]) for r in xs) / max(1, len(xs))
def play_avg(impl, key):
    xs = [r for r in play_runs if r["impl"] == impl]
    return sum(r[key] for r in xs) / max(1, len(xs))

# Side-by-side: baseline (light) vs your-tuned (saturated)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
labels = ["BROKEN", "FIXED"]

base_s = [M3_BASELINE["broken"]["success_rate"], M3_BASELINE["fixed"]["success_rate"]]
your_s = [play_succ("broken"), play_succ("fixed")]
xs = list(range(2)); width = 0.36
axes[0].bar([x - width/2 for x in xs], base_s, width=width, color=ICE,
            label=f"baseline (rate=0.4, stop=3)", zorder=2)
b = axes[0].bar([x + width/2 for x in xs], your_s, width=width,
                color=[CORAL, NAVY], label=f"your knobs", zorder=3)
for k, v in enumerate(your_s):
    axes[0].text(k + width/2, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")
axes[0].set_xticks(xs); axes[0].set_xticklabels(labels)
axes[0].set_ylim(0, 110); axes[0].set_ylabel("%")
axes[0].set_title("Success rate (baseline vs yours)"); axes[0].legend(fontsize=9)

base_t = [M3_BASELINE["broken"]["avg_tokens"], M3_BASELINE["fixed"]["avg_tokens"]]
your_t = [play_avg("broken", "tokens"), play_avg("fixed", "tokens")]
axes[1].bar([x - width/2 for x in xs], base_t, width=width, color=ICE)
b = axes[1].bar([x + width/2 for x in xs], your_t, width=width,
                color=[CORAL, NAVY])
for k, v in enumerate(your_t):
    axes[1].text(k + width/2, v + max(your_t) * 0.02, f"{v:.0f}",
                 ha="center", fontweight="bold")
axes[1].set_xticks(xs); axes[1].set_xticklabels(labels)
axes[1].set_title("Avg tokens (baseline vs yours)")

base_l = [M3_BASELINE["broken"]["avg_latency_s"], M3_BASELINE["fixed"]["avg_latency_s"]]
your_l = [play_avg("broken", "latency_ms")/1000, play_avg("fixed", "latency_ms")/1000]
axes[2].bar([x - width/2 for x in xs], base_l, width=width, color=ICE)
b = axes[2].bar([x + width/2 for x in xs], your_l, width=width,
                color=[CORAL, NAVY])
for k, v in enumerate(your_l):
    axes[2].text(k + width/2, v + max(your_l) * 0.02, f"{v:.1f}s",
                 ha="center", fontweight="bold")
axes[2].set_xticks(xs); axes[2].set_xticklabels(labels)
axes[2].set_title("Avg latency (baseline vs yours)")

fig.suptitle(f"Module 3 playground — your knobs: stop={STOP_AFTER}, "
             f"wait≤{MAX_WAIT_S}s, failure_rate={FAILURE_RATE}",
             fontsize=14, fontweight="bold", color=INK, y=1.02)
fig.tight_layout(); plt.show()


## 5. Module 4 — Observability and execution traces

Production agents that aren't *traceable* aren't really debuggable. The fix is one decorator's worth of code: a per-step span that records the tree of LLM calls and tool calls. We've been quietly emitting these throughout the notebook; in this module we put them to work on a single full agent run.

We render the resulting span tree as:
- an **ASCII timeline** (immediately readable in any terminal)
- a **Gantt chart** (one bar per span, parented by start order)

> The slide-worthy claim: *with a per-step trace, every failure mode from the previous modules becomes a SQL query against your trace store.* "Find runs where `tool.call.latency_ms > 30000`" finds your retry storms. "Find runs where `tool.call.args` contains the wrong city" finds your planner bugs.


In [ ]:
TASK_M4 = ("Find the customer name and city for the highest-amount delivered order, "
           "then tell me the weather in that city right now.")

tracer = Tracer("/tmp/exp4_trace.jsonl", run_id="exp4")

def run_traced():
    messages = [
        {"role":"system","content":"You are a data assistant. Tools: sql_query and get_weather. Answer in one short sentence after observing each tool result."},
        {"role":"user","content":TASK_M4},
    ]
    final = None
    with tracer.span("agent.run", task="exp4") as root:
        for step_idx in range(6):
            with tracer.span("agent.step", step_idx=step_idx) as step:
                with tracer.span("llm.call") as ls:
                    res = chat_with_tools(messages, tools=openai_tool_specs(), max_tokens=512)
                    ls.set("tokens_in", res.prompt_tokens)
                    ls.set("tokens_out", res.completion_tokens)
                    ls.set("tool_calls", len(res.tool_calls))
                if res.tool_calls:
                    messages.append({"role":"assistant","tool_calls":res.tool_calls})
                    for tc in res.tool_calls:
                        with tracer.span("tool.call", tool=tc["name"]) as ts:
                            ts.set("args", tc["arguments"])
                            result, telem = run_tool(tc["name"], tc["arguments"])
                            ts.set("validation_ok", telem["validation_ok"])
                        messages.append({"role":"tool","name":tc["name"],
                                         "content":json.dumps(result, default=str)})
                    continue
                final = (res.content or "").strip()
                break
        root.set("final_answer", final)
    return final

ans = run_traced()
print(f"Final answer: {ans}")
print(f"\nEmitted {len(tracer._all)} spans:")
for s in tracer._all:
    a = s["attributes"]
    extra = ""
    if s["name"] == "tool.call":
        extra = f"  tool={a.get('tool')} args={json.dumps(a.get('args'))}"
    elif s["name"] == "llm.call":
        extra = f"  in={a.get('tokens_in','?')}t out={a.get('tokens_out','?')}t"
    print(f"  {s['name']:<14s}  dur={s['duration_ms']:7.1f}ms {extra}")


In [ ]:
# === ASCII timeline + Gantt chart ===
spans = sorted(tracer._all, key=lambda s: s["start_ns"])
t0 = spans[0]["start_ns"]

print("=== ASCII timeline ===")
parents = {s["span_id"]: s.get("parent_id") for s in spans}
for s in spans:
    start = (s["start_ns"] - t0) / 1e9
    depth = 0; cur = s["span_id"]
    while parents.get(cur) is not None:
        depth += 1; cur = parents[cur]
    a = s.get("attributes") or {}
    extra = ""
    if s["name"] == "tool.call":
        extra = f"  tool={a.get('tool')}  args={a.get('args')}"
    elif s["name"] == "llm.call":
        extra = f"  in={a.get('tokens_in')}t out={a.get('tokens_out')}t"
    print(f"{start:6.2f}s  {'  '*depth}{s['name']:<14s} {s['duration_ms']:7.1f}ms{extra}")

# Gantt
color_for = {"agent.run": NAVY, "agent.step": INDIGO,
             "llm.call": TEAL, "tool.call": CORAL}
fig, ax = plt.subplots(figsize=(13, 5))
y_positions = list(range(len(spans)))[::-1]
for y, s in zip(y_positions, spans):
    start = (s["start_ns"] - t0) / 1e9
    dur = max(s["duration_ms"]/1000, 0.05)
    ax.barh(y, dur, left=start, color=color_for.get(s["name"], MUTED),
            edgecolor="white", linewidth=1, height=0.75, zorder=3)
    label = s["name"]
    a = s.get("attributes") or {}
    if s["name"] == "tool.call":     label = f"tool.call({a.get('tool')})"
    elif s["name"] == "agent.step":  label = f"agent.step idx={a.get('step_idx')}"
    elif s["name"] == "llm.call":
        label = f"llm.call  in={a.get('tokens_in','?')}t  out={a.get('tokens_out','?')}t"
    ax.text(start + dur + 0.1, y, label, va="center", fontsize=9, color=INK)
ax.set_xlabel("seconds (since trace start)")
ax.set_yticks(y_positions); ax.set_yticklabels([s["name"] for s in spans])
ax.set_title("Module 4 — agent execution trace (Gantt)",
             fontsize=14, fontweight="bold", color=INK)
ax.grid(True, axis="x", alpha=0.3, color="#E5E7EB", zorder=0)
fig.tight_layout(); plt.show()


**Takeaway — observability is one decorator's work.**

Look at the Gantt above. Every failure mode from Modules 1–3 is now a *query against this tree*:
- "Why is this slow?" → which `llm.call` or `tool.call` bar is widest?
- "Did the planner pick the right city?" → read the `tool.call args` line on the timeline.
- "Are we in a retry storm?" → count `tool.call` spans within one `agent.step`.

You didn't add a new dependency. You wrote a 90-line `Tracer` class and wrapped four call sites. **You earn this for free if you write the four lines from the start.**


## 6. Wrap-up — what you just built

Across this notebook, **you** built — on Colab's free T4, with no API keys —

1. **A function-calling agent loop** with strict Pydantic-validated tools and a SQLite + mock-weather backend.
2. **A router-worker state machine** with a Pydantic-typed classifier and forced single-tool-call workers, compared head-to-head with a free-form ReAct loop.
3. **A retry-storm comparison** between a naive try/except and a Tenacity-wrapped policy on a 40 %-flaky upstream — including a playground where you tuned the policy yourself and watched the bars move.
4. **An OpenTelemetry-style traced agent run** rendered as a Gantt chart you generated from your own spans.

### The one slide for the executives

> **Non-deterministic models do not absolve you from deterministic engineering.** Your agent's reliability is the product of your *planning structure*, your *retry policy*, your *memory management*, and your *observability* — none of which are properties of the model itself.

### Where to go next

| Topic | Public reference |
|---|---|
| LangGraph (production-grade graphs) | https://langchain-ai.github.io/langgraph/ |
| Plan-and-Execute pattern | https://blog.langchain.dev/planning-agents/ |
| Tenacity retry library | https://tenacity.readthedocs.io/ |
| Function-calling guide (OpenAI) | https://platform.openai.com/docs/guides/function-calling |
| ReAct (Yao et al., 2022) | https://arxiv.org/abs/2210.03629 |
| OpenTelemetry | https://opentelemetry.io/ |

### One last note

> *Naman Goyal is a Machine Learning Engineer at Google DeepMind. Personal views; synthesis of publicly available engineering practice. Not affiliated with or representative of Google DeepMind. All citations link to public documentation, open-source libraries, or arXiv.*
